# Dota 2 matches: loading, labelling and cleaning

72,915 public matches from the OpenDota API, collected with `opendota_matches.py`
(spread across 12 months, rotating the hour each day so all times of day are covered).

Steps: check for missing values, translate codes into readable game modes, lobby types
and server regions (OpenDota's constants, completed with the dotaconstants list on GitHub),
and remove matches under 5 minutes (85 matches that never really started).
Output: `data/dota_matches_clean.csv`, used in notebook 07.

In [18]:
import pandas as pd

d = pd.read_csv("data/dota_matches.csv", parse_dates=["start_time_utc"])
heroes = pd.read_csv("data/dota_heroes.csv")

print("Matches:", len(d), "| Heroes in list:", len(heroes))
print("From", d["start_time_utc"].min().date(), "to", d["start_time_utc"].max().date())
d.isna().mean().round(3)

Matches: 73000 | Heroes in list: 127
From 2025-10-21 to 2026-10-19


match_id          0.0
start_time_utc    0.0
duration_sec      0.0
lobby_type        0.0
game_mode         0.0
avg_rank_tier     0.0
num_rank_tier     0.0
cluster           0.0
radiant_win       0.0
radiant_heroes    0.0
dire_heroes       0.0
dtype: float64

In [19]:
print("Match length (minutes):")
print((d["duration_sec"] / 60).describe().round(1))

print("\nGame modes:");   print(d["game_mode"].value_counts().head(8))
print("\nLobby types:");  print(d["lobby_type"].value_counts())
print("\nServer clusters:", d["cluster"].nunique())
print("\nSkill level (avg_rank_tier):"); print(d["avg_rank_tier"].describe().round(1))
print("\nHeroes example:"); print(d[["radiant_heroes", "dire_heroes"]].head(3))

Match length (minutes):
count    73000.0
mean        37.3
std         12.0
min          0.0
25%         28.7
50%         36.8
75%         44.6
max        134.3
Name: duration_sec, dtype: float64

Game modes:
game_mode
22    49554
23    19615
4      1878
18      848
1       573
3       280
2       155
13       90
Name: count, dtype: int64

Lobby types:
lobby_type
7     45102
0     27071
4       571
9       105
14       90
1        61
Name: count, dtype: int64

Server clusters: 59

Skill level (avg_rank_tier):
count    73000.0
mean        41.8
std         16.7
min         11.0
25%         31.0
50%         42.0
75%         53.0
max         75.0
Name: avg_rank_tier, dtype: float64

Heroes example:
  radiant_heroes      dire_heroes
0  51|50|21|86|9   14|8|26|104|22
1  10|14|53|8|21  71|114|84|44|49
2  9|41|84|85|68   29|27|110|1|25


In [20]:
import requests

def constants(name):
    return requests.get(f"https://api.opendota.com/api/constants/{name}", timeout=30).json()

game_modes  = constants("game_mode")
lobby_types = constants("lobby_type")
clusters    = constants("cluster")   # cluster number -> region number
regions     = constants("region")    # region number -> region name

d["game_mode_name"]  = d["game_mode"].astype(str).map({str(v["id"]): v["name"] for v in game_modes.values()})
d["lobby_type_name"] = d["lobby_type"].astype(str).map({str(v["id"]): v["name"] for v in lobby_types.values()})
cluster_to_region = {c: regions.get(str(r)) for c, r in clusters.items()}
d["region"] = d["cluster"].astype(str).map(cluster_to_region)

print(d["game_mode_name"].value_counts().head(8))
print(d["lobby_type_name"].value_counts())
print(d["region"].value_counts())

game_mode_name
game_mode_all_draft         49554
game_mode_turbo             19615
game_mode_single_draft       1878
game_mode_ability_draft       848
game_mode_all_pick            573
game_mode_random_draft        280
game_mode_captains_mode       155
game_mode_limited_heroes       90
Name: count, dtype: int64
lobby_type_name
lobby_type_ranked        45102
lobby_type_normal        27071
lobby_type_coop_bots       571
lobby_type_battle_cup      105
lobby_type_new_player       90
lobby_type_practice         61
Name: count, dtype: int64
region
STOCKHOLM               23896
SINGAPORE               13410
EUROPE                   7835
AUSTRIA                  3006
PERU                     2905
PW TELECOM WUHAN         2738
US EAST                  2297
BRAZIL                   1884
PW UNICOM                1621
JAPAN                    1497
US WEST                   862
PW TELECOM ZHEJIANG       786
CHILE                     467
ARGENTINA                 350
PW TELECOM GUANGDONG      330
AU

In [21]:
minutes = d["duration_sec"] / 60
for limit in [1, 5, 10, 15]:
    print(f"Under {limit:>2} min: {(minutes < limit).sum():>5}  ({(minutes < limit).mean():.1%})")

Under  1 min:    83  (0.1%)
Under  5 min:    85  (0.1%)
Under 10 min:   365  (0.5%)
Under 15 min:   924  (1.3%)


In [22]:
print("Our cluster codes:", sorted(d["cluster"].unique())[:10])
print("Cluster list sample:", list(clusters.items())[:5])
print("Region list sample:", list(regions.items())[:5])

Our cluster codes: [np.int64(117), np.int64(118), np.int64(121), np.int64(122), np.int64(123), np.int64(129), np.int64(141), np.int64(142), np.int64(144), np.int64(145)]
Cluster list sample: [('111', 1), ('112', 1), ('113', 1), ('114', 1), ('117', 1)]
Region list sample: [('1', 'US WEST'), ('2', 'US EAST'), ('3', 'EUROPE'), ('5', 'SINGAPORE'), ('6', 'DUBAI')]


In [23]:
print(d["region"].value_counts())
print("\nMatches with an unknown region:", d["region"].isna().sum())
print("Clusters not in OpenDota's list:", sorted(d.loc[d["region"].isna(), "cluster"].unique()))

region
STOCKHOLM               23896
SINGAPORE               13410
EUROPE                   7835
AUSTRIA                  3006
PERU                     2905
PW TELECOM WUHAN         2738
US EAST                  2297
BRAZIL                   1884
PW UNICOM                1621
JAPAN                    1497
US WEST                   862
PW TELECOM ZHEJIANG       786
CHILE                     467
ARGENTINA                 350
PW TELECOM GUANGDONG      330
AUSTRALIA                 300
PW UNICOM TIANJIN         292
SOUTHAFRICA                73
DUBAI                      20
Name: count, dtype: int64

Matches with an unknown region: 8431
Clusters not in OpenDota's list: [np.int64(129), np.int64(189), np.int64(190), np.int64(381), np.int64(382), np.int64(404), np.int64(405), np.int64(406), np.int64(407), np.int64(410), np.int64(412), np.int64(413), np.int64(414), np.int64(415), np.int64(417), np.int64(423), np.int64(436)]


In [24]:
base = "https://raw.githubusercontent.com/odota/dotaconstants/master/build/"
gh_clusters = requests.get(base + "cluster.json", timeout=30).json()
gh_regions  = requests.get(base + "region.json", timeout=30).json()

gh_map = {c: gh_regions.get(str(r)) for c, r in gh_clusters.items()}
d["region_gh"] = d["cluster"].astype(str).map(gh_map)

print("Unknown with API list:   ", d["region"].isna().sum())
print("Unknown with GitHub list:", d["region_gh"].isna().sum())
print("\nStill unknown, by cluster:")
print(d.loc[d["region_gh"].isna(), "cluster"].value_counts())

Unknown with API list:    8431
Unknown with GitHub list: 3505

Still unknown, by cluster:
cluster
413    1491
417     654
415     509
414     319
423     128
436     122
129     104
190      55
406      50
405      31
404      26
407      16
Name: count, dtype: int64


In [25]:
# Regions: use the GitHub list, unknown servers labelled "Unknown"
d["region"] = d["region_gh"].fillna("Unknown")
d = d.drop(columns="region_gh")

# Readable labels: "game_mode_all_draft" -> "All Draft"
d["game_mode_name"]  = d["game_mode_name"].str.replace("game_mode_", "").str.replace("_", " ").str.title()
d["lobby_type_name"] = d["lobby_type_name"].str.replace("lobby_type_", "").str.replace("_", " ").str.title()

# Cleaning rule: matches under 5 minutes never really started
clean = d[d["duration_sec"] >= 5 * 60].copy()
print("Matches before:", len(d), "| after:", len(clean), "| removed:", len(d) - len(clean))

clean.to_csv("data/dota_matches_clean.csv", index=False)
print("Saved data/dota_matches_clean.csv")

Matches before: 73000 | after: 72915 | removed: 85
Saved data/dota_matches_clean.csv
